# Đề cập lại bài toán
- Bài toán ban đầu của nhóm là phân tích và dự đoán giá nhà ở 2 thành phố lớn là Hồ Chí Minh và Hà Nội nên nhóm sẽ xây dựng mô hình để dự đoán giá nhà


# Xây dựng mô hình

## Chuẩn bị dữ liệu chung cho cả 2 mô hình

In [90]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/unified/preprocessed_merged.csv')

df = df[(df['gia'] >= 0.5) & (df['gia'] <=500) & (df['so_tang'] <= 16) & (df['phong_ngu'] <= 30) & (df['phong_tam'] <= 30) & (df['dien_tich_dat'] <= 3000) & (df['dien_tich_dat'] > 5) & df['gia_tren_m2'] > 0]

Chia tập dữ liệu thành 80/20 với tương ứng với tập train/test 

In [91]:
# Giả sử cột target là 'price'
X = df.drop('gia', axis=1)
y = df['gia']

# Chia dữ liệu thành tập train và test với tỉ lệ 80-20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Mô hình 1
- Mô hình 1 sẽ dùng một biến kết hợp đó là quan x dien_tich_dat và cộng với diem_sinh_loi trong đó
- quan x dien_tich_dat là One hot encoding quận sau đó nhân với diện tích đất để ra biến mới, lý do kết hợp như này vì phần lớn giá nhà ở các trung tâm như Hà Nội và Hồ Chí Minh phụ thuộc rất nhiều vào vị trí, có những căn nhà vài mét vuông nhưng ở vị trí đắt đỏ lại có thể bán được vài tỷ [[1]](https://cafef.vn/con-duong-dat-do-bac-nhat-viet-nam-gia-nha-len-toi-3-ty-dong-m2-188240921095104835.chn) nên ý tưởng là thay vì chỉ quan tâm đến giá theo diện tích đất chung ta sẽ quan tâm đến giá theo diện tích đất theo từng quận sau đó cộng với diem_sinh_loi vì biến này được tính theo số phòng ngủ,số phòng tắm, số tầng,... vì thế cũng sẽ khái quát được độ lớn của ngôi nhà nên cũng sẽ ảnh hưởng đến giá.
- Do vậy 2 feature này sẽ tổng quát được vị trí theo giá và độ lớn của ngôi nhà



In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# --- Tạo feature như mô tả ---

# One-hot encoding cho "quan"
quan_dummies = pd.get_dummies(X_train['quan'], prefix='quan')
quan_dummies_test = pd.get_dummies(X_test['quan'], prefix='quan')


# Đảm bảo các cột quận giống nhau giữa train và test (tránh missing columns)
quan_dummies, quan_dummies_test = quan_dummies.align(quan_dummies_test, join='left', axis=1, fill_value=0)
# Đảm bảo các cột thành phố giống nhau giữa train và test (tránh missing columns)
thanh_pho_dummies, thanh_pho_dummies_test = thanh_pho_dummies.align(thanh_pho_dummies_test, join='left', axis=1, fill_value=0)

# feature: quan x dien_tich_dat
for col in quan_dummies.columns:
    X_train[f"{col}_x_dien_tich_dat"] = quan_dummies[col] * X_train['dien_tich_dat']
    X_test[f"{col}_x_dien_tich_dat"] = quan_dummies_test[col] * X_test['dien_tich_dat']

# Thêm các feature one-hot cho thanh_pho
for col in thanh_pho_dummies.columns:
    X_train[col] = thanh_pho_dummies[col]
    X_test[col] = thanh_pho_dummies_test[col]

# feature đầu ra cuối cùng: tất cả các "quan x dien_tich_dat" + diem_sinh_loi + các feature phap_ly + các feature thanh_pho
feature_cols = [c for c in X_train.columns if '_x_dien_tich_dat' in c]
feature_cols.append('diem_sinh_loi')

X_train_features = X_train[feature_cols].copy()
X_test_features = X_test[feature_cols].copy()

# --- So sánh mô hình ---

results = {}
from sklearn.metrics import mean_absolute_error

# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_features, y_train)
y_pred_lr = lin_reg.predict(X_test_features)
results['Linear Regression'] = {
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_lr)),
    'MAE': mean_absolute_error(y_test, y_pred_lr),
    'R2': r2_score(y_test, y_pred_lr)
}

# Random Forest
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_features, y_train)
y_pred_rf = rf.predict(X_test_features)
results['Random Forest'] = {
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_rf)),
    'MAE': mean_absolute_error(y_test, y_pred_rf),
    'R2': r2_score(y_test, y_pred_rf)
}

# Hiển thị kết quả
print("So sánh 2 thuật toán:")
for model_name, metrics in results.items():
    print(f"{model_name}:")
    print(f"    RMSE: {metrics['RMSE']:.2f}")
    print(f"    MAE:  {metrics['MAE']:.2f}")
    print(f"    R2:   {metrics['R2']:.4f}")



So sánh 2 thuật toán:
Linear Regression:
    RMSE: 13.59
    MAE:  6.61
    R2:   0.6830
Random Forest:
    RMSE: 14.61
    MAE:  5.39
    R2:   0.6337
